### 5. Feature Engineering

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/processed/deliveries_with_target.csv")

In [3]:
df.shape

(43648, 23)

In [4]:
df.columns

Index(['Order_ID', 'Agent_Age', 'Agent_Rating', 'Store_Latitude',
       'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude', 'Order_Date',
       'Order_Time', 'Pickup_Time', 'Weather', 'Traffic', 'Vehicle', 'Area',
       'Delivery_Time', 'Category', 'Order_DateTime', 'Pickup_DateTime',
       'Pickup_Next_Day', 'Distance', 'Extreme_Distance_Flag',
       'Night_Order_Flag', 'Minutes_per_km'],
      dtype='object')

In [5]:
df["Category"].value_counts(normalize=True)

Category
On-Time    0.501787
At Risk    0.298914
Delayed    0.199299
Name: proportion, dtype: float64

#### Feature Creation

In [12]:
order_time_dt = pd.to_datetime(
    df["Order_Time"],
    format="%H:%M:%S",
    errors="coerce"
)

df["Order_Hour"] = order_time_dt.dt.hour


In [17]:
df["Order_Hour"].value_counts().sort_index()

Order_Hour
0      429
8     1810
9     1934
10    1980
11    1950
12     888
13     779
14     785
15     871
16     706
17    4247
18    4467
19    4579
20    4504
21    4663
22    4558
23    4498
Name: count, dtype: int64

In [18]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df["Weekend_Flag"] = df["Order_Date"].dt.weekday.isin([5, 6]).astype(int)

In [20]:
df["Weekend_Flag"].value_counts()


Weekend_Flag
0    31627
1    12021
Name: count, dtype: int64

In [21]:
df["Peak_Hour_Flag"] = df["Order_Hour"].isin(
    [8, 9, 10, 11, 17, 18, 19, 20]
).astype(int)

In [22]:
df["Peak_Hour_Flag"].value_counts()

Peak_Hour_Flag
1    25471
0    18177
Name: count, dtype: int64

In [24]:
pd.crosstab(df["Order_Hour"], df["Night_Order_Flag"])

Night_Order_Flag,0,1
Order_Hour,,
0,0,429
8,1810,0
9,1934,0
10,1980,0
11,1950,0
12,888,0
13,779,0
14,785,0
15,871,0


In [25]:
pd.crosstab(df["Peak_Hour_Flag"], df["Category"], normalize="index")

Category,At Risk,Delayed,On-Time
Peak_Hour_Flag,,,
0,0.328162,0.075810,0.596028
1,0.278042,0.287425,0.434533


In [27]:
df["Distance_Bucket"] = pd.cut(
    df["Distance"],
    bins=[0, 5, 10, 20, 50, 100, 300, np.inf],
    labels=["0–5", "5–10", "10–20", "20–50", "50–100", "100–300", "300+"]
)

In [28]:
df["Traffic_Weather"] = df["Traffic"].astype(str) + "_" + df["Weather"].astype(str)

In [29]:
df["Distance_Vehicle"] = df["Distance_Bucket"].astype(str) + "_" + df["Vehicle"].astype(str)

In [31]:
categorical_cols = [
    "Weather",
    "Traffic",
    "Vehicle",
    "Area",
    "Distance_Bucket",
    "Traffic_Weather",
    "Distance_Vehicle"
]

numerical_cols = [
    "Agent_Age",
    "Agent_Rating",
    "Distance",
    "Order_Hour",
    "Peak_Hour_Flag",
    "Weekend_Flag",
    "Night_Order_Flag",
    "Extreme_Distance_Flag"
]

In [32]:
df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

In [33]:
df.columns

Index(['Order_ID', 'Agent_Age', 'Agent_Rating', 'Store_Latitude',
       'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude', 'Order_Date',
       'Order_Time', 'Pickup_Time', 'Weather', 'Traffic', 'Vehicle', 'Area',
       'Delivery_Time', 'Category', 'Order_DateTime', 'Pickup_DateTime',
       'Pickup_Next_Day', 'Distance', 'Extreme_Distance_Flag',
       'Night_Order_Flag', 'Minutes_per_km', 'Order_Hour', 'Weekend_Flag',
       'Peak_Hour_Flag', 'Distance_Bucket', 'Traffic_Weather',
       'Distance_Vehicle'],
      dtype='object')

In [34]:
df_encoded = df_encoded.drop(
    columns=['Order_ID', 'Store_Latitude',
       'Store_Longitude', 'Drop_Latitude', 'Drop_Longitude', 'Order_Date',
       'Order_Time', 'Pickup_Time',
       'Delivery_Time',  'Order_DateTime', 'Pickup_DateTime',
       'Minutes_per_km' ],
    errors="ignore"
)

In [36]:
df_encoded.head()

,Agent_Age,Agent_Rating,Category,Pickup_Next_Day,Distance,Extreme_Distance_Flag,Night_Order_Flag,Order_Hour,Weekend_Flag,Peak_Hour_Flag,...,Distance_Vehicle_10–20_van,Distance_Vehicle_20–50_motorcycle,Distance_Vehicle_20–50_scooter,Distance_Vehicle_20–50_van,Distance_Vehicle_300+_motorcycle,Distance_Vehicle_300+_scooter,Distance_Vehicle_300+_van,Distance_Vehicle_5–10_motorcycle,Distance_Vehicle_5–10_scooter,Distance_Vehicle_5–10_van
0,37,4.9,Delayed,0,3.025149,0,0,11,1,1,...,False,False,False,False,False,False,False,False,False,False
1,34,4.5,On-Time,0,20.183530,0,0,19,0,1,...,False,False,True,False,False,False,False,False,False,False
2,23,4.4,Delayed,0,1.552758,0,0,8,1,1,...,False,False,False,False,False,False,False,False,False,False
3,38,4.7,On-Time,0,7.790401,0,0,18,0,1,...,False,False,False,False,False,False,False,True,False,False
4,32,4.6,At Risk,0,6.210138,0,0,13,1,0,...,False,False,False,False,False,False,False,False,True,False


In [37]:
df_encoded.shape

(43648, 66)

In [40]:
df_encoded.isna().sum().sum()

np.int64(0)

In [41]:
df_encoded.to_csv("../data/processed/deliveries_features_ready.csv", index=False)